In [ ]:
# /content/drive/MyDrive/rokey/AI/application/6/Tuebingen_Neckarfront.jpg

# /content/drive/MyDrive/rokey/AI/application/6/vangogh_starry_night.jpg

In [3]:
# PyTorch의 핵심 라이브러리를 불러옴.
import torch
# PyTorch의 자동 미분 기능(Autograd)을 위한 Variable 클래스를 불러옴. (최신 PyTorch에서는 텐서가 대체함)
from torch.autograd import Variable # 파이토치의 자동미분
# 신경망 레이어(nn) 모듈을 불러옴.
import torch.nn as nn
# 함수형 신경망 연산(F) 모듈을 불러옴.
import torch.nn.functional as F
# 옵티마이저(optim) 모듈을 불러옴.
from torch import optim

# TorchVision 라이브러리를 불러옴.
import torchvision
# 이미지 변환(transforms) 모듈을 불러옴.
from torchvision import transforms

# PIL 라이브러리의 Image 모듈을 불러옴. 이미지 처리에 사용함.
from PIL import Image
# 순서가 보장되는 딕셔너리(OrderedDict) 클래스를 불러옴. (특정 구조 저장에 유용함)
from collections import OrderedDict

🌈 왜 CNN 히든 레이어가 필요하냐?

스타일 트랜스퍼에서
스타일 = 이미지의 "채널 간 상관도"
이걸 계산하려면 CNN이 추출한 feature map이 필요해.

이미지 원본의 픽셀 자체로는 “붓질/패턴/질감” 같은 스타일 정보를 잡을 수 없음.

그래서 보통 이렇게 함:

① 미리 학습된 VGG19 모델 불러오기
② 원하는 레이어(conv1_1, conv2_1, conv3_1…)의 출력(feature map)을 가져오기
③ 그걸 GramMatrix에 넣기

그러면 그 레이어의 feature map이 스타일 표현이 되는 것.

In [4]:
# Gram Matrix 계산하는 파이토치 모듈 정의
class GramMatrix(nn.Module):
  # 순전파 정의
  def forward(self, input):
    b,c,h,w = input.size()
    # input: hidden layer에서 출력한 Feature map에서 가져온 값
    # >> 특징(feature) 추출한 값 = 스타일 표현 사용했으니깐
    # 입력 텐서 배치(b), 채널(c), 높이(h), 너비(w) (차원정보 추출)
    # (1,64,256,256) b=1(1장), c=64(64개의 채널=Feature map), h=256(256px 높이), w=256(256px 너비)
    F = input.view(b, c, h*w) # view=reshape
    # 4차원 >> 3차원 변경 (텐서를 배치*채널, 높이*너비) 형태로 flatten 하는데,
    # 이 구현에서는 이미지의 공간적 크기(H,W)를 평탄화한 것(채널은 질감 등 특징 있으니까 hw를 평탄화, b=1이라 신경x)
    # (1,64,256,256) >> (1,64,256*256) 3D >> 2D 이미지
    # c * (h*w) >> 공간 차원을 펼친 Feature map 생성
    # 채널1(질감)과 채널2(색상)이 서로 얼마나 관계있는지
    # 높으면 두 특징이 자주 함께 나타남
    # bmm은 배치단위 행렬곱이므로 (c , h*w) 와 (c, h*w)간의 inner product를 위해 전치가 필요합니다.
    # (2,64,32,32) >> (2,64,32*32) 2d 이미지
    # 배치(2)마다(각각의 이미지마다) 64개 채널(feature map)*1024개 위치 값
    # 채널별(색상, 질감..)로 길이가 1024개 있는 벡터
    # c*(h*w) >> 공간 차원을 펼친 Feature map 생성
    # c*(h*w), c*(h*w) 내적하려면 차원을 맞춰주기 위해 전처리
    # 여러개의 필터(채널수만큼)로 여러 특징맵(채널수만큼) 뽑으니 그 특징맵 크기 h * w 를 하나로 쭉 늘려서 c x (h*w) 행렬을 뽑아
    # 전치 내적하면 결국엔 특징맵들끼리 곱해서 더하는 꼴이니 교본에 나온 G를 구하는 수식이 완성

    G = torch.bmm(F, F.transpose(1,2))
    # 배치 행렬 곱(bmm) 사용, Gram matrix 만듦
    # F와 F의 채널-공간 차원 전치(transpose) 곱해줌
    # 각 채널 벡터들을 표(테이블)처럼 i×j 조합으로 싹 다 내적해서 정리한 "요약표"
    # F shape: [b, c, HW]
    # F.transpose(1,2): [b, HW, c] : 행렬곱하려면 사이즈 2*3x2*3 -> 2*3x3*2(transpose)
    # 결과의 크기 (b, c, c)
    # 채널 1.shape (b, n, m) / 채널 2.shape (b, m, p)
    # >> 배치 차원 b개에 대해 각각의 행렬 곱 수행
    # >> (b, n, p)
    '''
          채널0  채널1  채널2  ...
    채널0   G00   G01   G02
    채널1   G10   G11   G12
    채널2   G20   G21   G22
    ...

    '''

    G.div_(h*w)
    # gram matrix를 높이*너비(h*w)로 나누어 정규화
    # 왜? 값의 범위를 안정화하기 위해(정규화 >> 안정)
    # 왜 (높이*너비)? 이미지 크기에 관계없이 일정한 값 갖게 하기 위해서

    return G

In [5]:
# gram matrix에 대해 MSE(평균 제곱 오차) 손실 계산하는 모듈 정의
class GramMSELoss(nn.Module):
  # 순전파(forward) 정의함. input: 현재 이미지의 특징, target: 목표로 하는 이미지의 Gram matrix
  def forward(self, input, target):
    out = nn.MSELoss()(GramMatrix()(input),target)
    # GramMatrix 모듈 통과 >> gram matrix 계산 >> 목표(target) gram matrix와 비교
    # >> MSE 손실 계산
    # target은 함수가 아니라서 (target)으로 묶지 않는다. tensor 데이터다.
    return (out)

In [ ]:
'''
out = nn.MSELoss()(GramMatrix()(input),target)
mse = nn.MSELoss() # 객체 생성
gm_input = GramMatrix()(input)
out = mse(gm_input, target)
'''
# target도 style 정보(gram matrix)이다.
# __call__ 쓸때 객체를 함수로 쓸수 있는데, nn.Module 에서는 foward() 함수가 실행되도록 정의해 놓았

Content-style Loss

In [7]:
class VGG(nn.Module):
    def __init__(self, pool='max'):
        super(VGG, self).__init__()
        #vgg modules
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_4 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_4 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_4 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        if pool == 'max':
            self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)
        elif pool == 'avg':
            self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool3 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool4 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool5 = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x, out_keys):
        out = {}
        out['r11'] = F.relu(self.conv1_1(x))
        out['r12'] = F.relu(self.conv1_2(out['r11']))
        out['p1'] = self.pool1(out['r12'])
        out['r21'] = F.relu(self.conv2_1(out['p1']))
        out['r22'] = F.relu(self.conv2_2(out['r21']))
        out['p2'] = self.pool2(out['r22'])
        out['r31'] = F.relu(self.conv3_1(out['p2']))
        out['r32'] = F.relu(self.conv3_2(out['r31']))
        out['r33'] = F.relu(self.conv3_3(out['r32']))
        out['r34'] = F.relu(self.conv3_4(out['r33']))
        out['p3'] = self.pool3(out['r34'])
        out['r41'] = F.relu(self.conv4_1(out['p3']))
        out['r42'] = F.relu(self.conv4_2(out['r41']))
        out['r43'] = F.relu(self.conv4_3(out['r42']))
        out['r44'] = F.relu(self.conv4_4(out['r43']))
        out['p4'] = self.pool4(out['r44'])
        out['r51'] = F.relu(self.conv5_1(out['p4']))
        out['r52'] = F.relu(self.conv5_2(out['r51']))
        out['r53'] = F.relu(self.conv5_3(out['r52']))
        out['r54'] = F.relu(self.conv5_4(out['r53']))
        out['p5'] = self.pool5(out['r54'])
        return [out[key] for key in out_keys]

[VGG 구조 패턴]

- Block 1: 3→64→64 (얕은 특징: 선, 모서리)
- Block 2: 64→128→128 (중간 특징: 질감)
- Block 3: 128→256→256→256 (깊은 특징: 패턴)
- Block 4: 256→512→512→512 (더 복잡한 특징)
- Block 5: 512→512→512→512 (추상적 특징)

어떤 블록까지 가져올거냐에 따라 균형이 달라짐

In [18]:
vgg = VGG()

# 로드할 이미지 파일 이름들을 정의함. ('vangogh_starry_night.jpg', 'Tuebingen_Neckarfront.jpg')
img2 = '/content/drive/MyDrive/rokey/AI/application/6/Tuebingen_Neckarfront.jpg'
img1 ='/content/drive/MyDrive/rokey/AI/application/6/vangogh_starry_night.jpg'

# 이미지 디렉토리와 파일 이름을 결합하여 PIL Image 객체 리스트로 로드함.
img1 = Image.open(img1)
img2 = Image.open(img2)
imgs = []
imgs.append(img1)
imgs.append(img2)

img_size = 512
prep = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), # Tensor (chw) [0,1] 범위
        transforms.Lambda(lambda x: x[torch.LongTensor([2,1,0])]), # RGB >> BGR
        transforms.Normalize(mean=[0.40760392, 0.45795686, 0.48501961], #subtract imagenet mean
                                                    std=[1,1,1]),
        transforms.Lambda(lambda x: x.mul_(255)) # pixel 범위 [0,255] 범위로 표현 >> VGG 입력 스케일로 복원
    ])

# 각 PIL Image 객체에 사전 정의된 전처리함수(prep)를 적용 >> 텐서로 변환
imgs_torch = [prep(img) for img in imgs] # imgs 이미지 리스트 전체

# GPU(CUDA) 사용이 가능하다면:
if torch.cuda.is_available():
    # 각 텐서에 배치 차원(unsqueeze(0))을 추가하고 GPU로 이동시킨 후, Variable로 감싸서 저장함.
    imgs_torch = [Variable(img.unsqueeze(0).cuda()) for img in imgs_torch]
# GPU를 사용할 수 없다면:
else:
    # 각 텐서에 배치 차원만 추가하고 Variable로 감싸서 저장함.
    imgs_torch = [Variable(img.unsqueeze(0)) for img in imgs_torch]
'''
if torch.cuda.is_available():
    # CUDA가 사용 가능하면, 텐서를 GPU로 이동 (.cuda() 또는 .to(device))
    imgs_torch = [img.unsqueeze(0).cuda() for img in imgs_torch]
else:
    # CUDA를 사용할 수 없으면, 텐서를 CPU에 유지
    imgs_torch = [img.unsqueeze(0) for img in imgs_torch]
와 같음!
'''

# 이미지 텐서 리스트를 스타일 이미지와 콘텐츠 이미지 변수에 할당
style_image, content_image = imgs_torch

# 형태 유지하면서 스타일만 변형되도록 학습(경사하강법) 시작 >> 이미지 자체가 학습대상(파라미터)
# (**) style transfer(전이) CNN 가중치 학습은 하지 않아요. opt_img의 픽셀값만 학습
opt_img = Variable(content_image.data.clone(), requires_grad=True) # content representation

'''
content_image.clone().detach().requires_grad_(True)
# content_image (데이터, 값) 복제(clone)해서 기존 그래프 분리(detach) >> 기울기 계산(requires_grad_(True))
: 이걸 더 많이 쓴다
'''

'\ncontent_image.clone().detach().requires_grad_(True)\n# content_image (데이터, 값) 복제(clone)해서 기존 그래프 분리(detach) >> 기울기 계산(requires_grad_(True))\n: 이걸 더 많이 쓴다 \n'

In [22]:
import torch
import torch.nn as nn
# GramMSELoss와 vgg는 정의되어 있다고 가정합니다.
# from your_modules import GramMSELoss, vgg, GramMatrix

# 사용할 GPU를 설정하고, VGG 모델을 GPU로 이동합니다.
# VGG 모델 인스턴스 생성

if torch.cuda.is_available():
    # VGG 모델을 GPU 메모리로 이동
    vgg = vgg.cuda()
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# 스타일 손실을 계산할 VGG 레이어 이름 정의
style_layers = ['r11','r21','r31','r41', 'r51']
# style_layers 의미? VGG 내부의 특정 합성곱 레이어

# 콘텐츠 손실을 계산할 VGG 레이어 이름 정의
# (일반적으로 중간정도 레이어 중 하나 사용)
content_layers = ['r42']

loss_layers = style_layers + content_layers
# 사용할 모든 손실 레이어를 스타일 레이어와 콘텐츠 레이어 합쳐 정의

# 각 스타일에 대해 GramMSELoss 모듈을 사용하고, 콘텐츠에 대해 MSELoss 모듈을 사용하도록 리스트를 정의함.
# [GramMSELoss, GramMSELoss, ..., nn.MSELoss] 형태가 됨.
# GramMSELoss : 스타일 손실 계산
# GramMSELoss가 nn.Module을 상속한다고 가정합니다.
loss_fus = [GramMSELoss()] * len(style_layers) + [nn.MSELoss()] * len(content_layers)

# GPU (CUDA) 사용이 가능하다면, 모든 손실 함수 모듈을 GPU 메모리로 이동시킴.
if torch.cuda.is_available():
    # loss_fns 리스트의 요소들을 새로운 모듈 인스턴스로 만들고 .cuda()를 적용
    # 리스트 복사를 방지하고 정확하게 GPU로 이동하기 위해 수정
    loss_fns = [GramMSELoss().to(device) for _ in style_layers] + \
               [nn.MSELoss().to(device) for _ in content_layers]
    # for _ in style_layers : style layers 개수만큼 GramMSELoss()
    # >> 각각 새로 생성
else:
    # CPU 사용 시에도 동일한 로직으로 인스턴스화
    loss_fns = [GramMSELoss().to(device) for _ in style_layers] + \
               [nn.MSELoss().to(device) for _ in content_layers]

# 스타일 레이어 수만큼 스타일 손실 모듈GramMSELoss + 컨텐츠 레이어 수만큼 컨텐츠 손실 모듈MSELoss
# 스타일 손실에 적용할 가중치(beta)를 정의
# 깊은 레이어일수록 (복잡한 패턴을 추출하는 레이어) 낮은 가중치를 주는 경향이 있음
# 왜? 가중치가 감쇠되니까
style_weights = [1e3/n**2 for n in [64, 128, 256, 512, 512]]
# [64, 128, 256, 512, 512] : 채널(c) 수
# 1e3/n**2 : 보정 값, 깊은 레이어일수록 feature 수가 많고 값도 커진다.


# 콘텐츠 손실에 부여할 가중치(alpha)
content_weights = [1e0] # 1e0 = 1

weights = style_weights + content_weights

# 최적화 목표값(style target) 계산
# style_image >> VGG 통과 >> 각 스타일 레이어 Gram Matrix 계산
# >> 변화도 추적에서 제외 (detach)
style_targets = [GramMatrix()(A).detach() for A in vgg(style_image, style_layers)]

# vgg(style_image, style_layers) : 지정한 레이어들의 특징맵(feature map) 리스트로 나옴
# [A_r11, A_r21,...] 여기서 A.shape(b,c,h,w)
# GramMatrix(A) >> (b,c,c) >> 스타일 표현
# .detach() 계산 그래프에서 분리시킴(역전파할 때 gradient 계산되지 않도록)
# >> 왜? style_targets 는 고정된 값(Ground truth)
# style_targets? 각 스타일 레이어에 대한 스타일 이미지 Gram Matrix 목록

# 최적화 목표값(content targets)을 계산함.
# 콘텐츠 이미지(content_image)를 VGG에 통과시켜 콘텐츠 레이어의 특징 맵을 추출하고 변화도 추적에서 제외(detach)했음.
# content_image 또한 이미 .cuda() 또는 .to(device)로 GPU에 로드되어 있다고 가정합니다.
content_targets = [A.detach() for A in vgg(content_image, content_layers)]

# 최종적으로 사용할 모든 목표값 리스트를 정의함.
targets = style_targets + content_targets

In [23]:
input_image = content_image.clone().requires_grad_(True)
# input_image는 VGG에 입력될 초기 이미지, 콘텐츠 이미지와 동일해야 함
input_image

tensor([[[[  15.0610,   19.0610,   32.0610,  ...,   68.0610,   71.0610,
             70.0610],
          [  10.0610,   16.0610,   16.0610,  ...,   69.0610,   70.0610,
             69.0610],
          [   6.0610,   15.0610,   23.0610,  ...,   70.0610,   71.0610,
             70.0610],
          ...,
          [ -81.9390,  -83.9390,  -84.9390,  ...,    5.0610,   -2.9390,
             -0.9390],
          [ -82.9390,  -85.9390,  -85.9390,  ...,   -8.9390,   -2.9390,
             -5.9390],
          [ -83.9390,  -88.9390,  -87.9390,  ...,   -7.9390,   -6.9390,
            -15.9390]],

         [[ -33.7790,  -32.7790,  -21.7790,  ...,   -2.7790,    0.2210,
             -0.7790],
          [ -33.7790,  -30.7790,  -34.7790,  ...,   -1.7790,   -0.7790,
             -1.7790],
          [ -31.7790,  -26.7790,  -21.7790,  ...,   -0.7790,    0.2210,
             -0.7790],
          ...,
          [ -91.7790,  -93.7790,  -94.7790,  ...,   19.2210,   11.2210,
             14.2210],
          [ -92.77

In [25]:
# L-BFGS 옵티마이저: 최적화 단계에서 closure 함수 요구

# input_image 초기 이미지(최적화 대상), CNN 가중치가 아니라, ***이미지 픽셀*** 학습
optimizer = torch.optim.LBFGS([input_image], max_iter=1)
# optimizer = torch.optim.Adam([input_image], lr=0.01)

# 최적화 함수 세기 위한 변수
n_iter = 0

# [추가된 코드] 콘텐츠 목표값 계산 및 전체 목표값 리스트 생성
# 1. 콘텐츠 이미지에서 특징 맵 추출 (목표 형태)
content_targets = [A.detach() for A in vgg(content_image, content_layers)]

# 2. 스타일 목표(style_targets)와 콘텐츠 목표(content_targets)를 합침
# 이 순서는 loss_layers = style_layers + content_layers 순서와 일치해야 함
targets = style_targets + content_targets

In [26]:
def closure():
    global n_iter

    # 이전 기울기를 초기화합니다.
    optimizer.zero_grad()

    # 생성된 이미지(input_image)를 VGG에 통과시켜 특징 맵을 추출합니다.
    # vgg는 이전에 정의되어 GPU로 이동되었다고 가정합니다.
    out = vgg(input_image, loss_layers)
    # input_image : 현재 생성된 이미지 >> vgg() >> feature map 생성
    # loss_layers에 해당하는 레이어의 feature map list가 나온다.

    # 총 손실을 계산합니다.
    layer_losses = [] # 스타일/콘텐츠 손실(개별적으로 저장)
    total_loss = 0

    for i, weight in enumerate(weights):
        target = targets[i] # 스타일 이미지 / 콘텐츠 이미지 target 값
        feature = out[i] # 현재 생성 이미지의 Feature map
        loss_fn = loss_fns[i] # 해당 레이어 loss function(GramMSELoss: style, MSELoss: content)

        # GramMatrix 계산이 필요한 스타일 레이어 처리 (GramMSELoss가 GramMatrix를 내부에서 처리한다고 가정)
        # GramMatrix()가 별도 모듈이면, GramMSELoss 내부에 GramMatrix가 포함되어 있어야 합니다.

        loss = weight * loss_fn(feature, target)
        layer_losses.append(loss.item())
        total_loss += loss

    # 역전파를 수행하여 기울기를 계산합니다.
    # gradient 계산됨 >> input_image에 대해 계산됨
    total_loss.backward()

    # 진행 상황을 출력합니다.
    if n_iter % 50 == 0:
        print(f"Iteration {n_iter}: Total Loss = {total_loss.item():.4f}")
        # print(f"Layer Losses: {layer_losses}") # 디버깅용

    n_iter += 1
    return total_loss

# 최적화 실행 (반복 횟수 지정)
num_iterations = 500 # 원하는 반복 횟수를 설정합니다.
for i in range(num_iterations):
    # L-BFGS는 step() 호출 시마다 클로저를 여러 번 호출할 수 있습니다.
    optimizer.step(closure)
    # closure 반환 값을 기반으로 다음 업데이트 방향 계산

    # 참고: Adam을 사용한다면, 루프는 다음과 같습니다.
    # loss = closure()
    # optimizer.step()

# 최종 결과 이미지 후처리 (옵션)
# 생성된 이미지를 [0, 1] 범위로 클리핑하여 픽셀 값을 보정합니다.
input_image.data.clamp_(0, 1)
# 최적화된 input_image.data가 최종 스타일 트랜스퍼 결과입니다.

Iteration 0: Total Loss = 1.4664
Iteration 50: Total Loss = 1.4663
Iteration 100: Total Loss = 1.4663
Iteration 150: Total Loss = 1.4662
Iteration 200: Total Loss = 1.4662
Iteration 250: Total Loss = 1.4661
Iteration 300: Total Loss = 1.4661
Iteration 350: Total Loss = 1.4660
Iteration 400: Total Loss = 1.4660
Iteration 450: Total Loss = 1.4659


tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.]]]])

## Style Transfer (스타일 전이) 정리
1. Target : 학습이 도달해야 할 기준 (Reference 정답)
- 생성되는 이미지(업데이트되는 이미지)가 따라가야 할 목표 상태(정답)
- 이 target은 이미지 자체가 아니라 Content 이미지의 Feature Map
Style 이미지의 Gram Matrix
로 정의됨.

2. Content 목표 / Style 목표

- Content 목표
  - 목적: 이미지의 형태·구조 유지
  - Target: 콘텐츠 이미지에서 추출한 Feature Map

- Style 목표

  - 목적: 스타일 이미지의 질감·색감·패턴 재현

  - Target: 스타일 이미지에서 추출한 Gram Matrix

3. 특징 정보 (Features)

- Content 이미지 → Feature Map (구조 정보)

- Style 이미지 → Gram Matrix (질감/패턴 정보)

    생성 이미지는 학습을 통해 이 두 target 값과 가까워지도록 업데이트됨.

4. Loss의 의미

- 생성 이미지 vs Content Target → Content Loss

- 생성 이미지 vs Style Target → Style Loss

    Loss는 두 target과의 차이를 의미하며 Loss가 0에 가까울수록 형태는 원본 콘텐츠처럼 되고 질감/색감은 스타일 이미지처럼 된다.